# 04. Validation ML et fine-tuning

**Objectif** : Valider les annotations LLM par des modèles ML supervisés (embeddings + LogisticRegression, option fine-tuning CamemBERT). Mesurer la corrélation LLM vs ML et la performance par bloc/arène.

## 0. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from config import CORPUS_V3, RESULTS_DIR, FIGURES_DIR, COLORS
warnings.filterwarnings("ignore")

FIG_DIR = FIGURES_DIR
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results: {RESULTS_DIR}")

## 4.1 Exécution du pipeline ML (embeddings)

Lancer en ligne de commande : `python src/ml_pipeline.py --mode embeddings`

Ou exécuter les cellules ci-dessous pour intégrer l'analyse.

In [ ]:
# Chargement des métriques si le pipeline a déjà été exécuté
emb_metrics = RESULTS_DIR / "ml_embeddings_metrics.csv"
val_report = RESULTS_DIR / "ml_validation_par_bloc_arena.csv"

if emb_metrics.exists():
    m = pd.read_csv(emb_metrics)
    print("Métriques embeddings + LR :")
    display(m[["accuracy", "f1_macro", "correlation_llm"]])
else:
    print("Exécuter : python src/ml_pipeline.py --mode embeddings")

## 4.2 Corrélation LLM vs ML par bloc et arène

In [ ]:
if val_report.exists():
    rep = pd.read_csv(val_report)
    fig, ax = plt.subplots(figsize=(8, 4))
    by_dim = rep.groupby("dimension")
    x = np.arange(len(rep))
    colors = [COLORS.get(r["level"], "#888") if r["dimension"] == "bloc" else "#666" for _, r in rep.iterrows()]
    ax.bar(x, rep["correlation_llm_ml"], color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels(rep["level"], rotation=45, ha="right")
    ax.set_ylabel("Corrélation LLM vs ML")
    ax.set_title("Validation croisée : accord annotations LLM / modèle embeddings")
    ax.axhline(0, color="gray", ls="--", lw=0.8)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig_ml_validation_bloc_arena.png", dpi=150, bbox_inches="tight")
    plt.show()
    display(rep)
else:
    print("Exécuter : python src/ml_pipeline.py --mode validate")

## 4.3 Fine-tuning CamemBERT (optionnel)

Pour fine-tuner un modèle dédié au stance :

```bash
python src/ml_pipeline.py --mode finetune --epochs 3
```

Nécessite GPU recommandé. Produit `data/models/camembert_stance/`.

## Synthèse

- Les embeddings (paraphrase-multilingual) + LogisticRegression offrent une validation indépendante des annotations LLM.
- Une corrélation > 0.7 entre prédictions ML et LLM renforce la validité du corpus.
- Le fine-tuning CamemBERT permet une inférence locale sans appel API.